# FairyZero — khảo sát trần hiệu năng thật sự của T4  *(bản 2)*

**Sửa so với bản 1:** lần chạy trước `CUDAExecutionProvider` im lặng rơi về CPU,
nên mọi ô đều trống. Hai nguyên nhân, cả hai đã xử lý:

1. **Thiếu đường dẫn thư viện CUDA.** CUDA EP `dlopen` cuDNN/cuBLAS *lúc nạp*.
   Trên Colab chúng nằm trong các gói pip `nvidia-*` của torch. Python **không
   thể** tự sửa `LD_LIBRARY_PATH` cho chính tiến trình nó đang chạy — phải đặt
   trước khi khởi động. Đây đúng là lý do engine cần `run.sh`.
2. **Sai phiên bản ORT.** Colab cài **1.30.0**, nhưng engine build với **1.20.1**.
   Đo 1.30 thì không nói lên gì về bản engine dùng.

Giờ phần đo chạy qua `bench_ort.sh` — nó ghim ORT 1.20.1 và dựng
`LD_LIBRARY_PATH` trước khi gọi Python.

Bộ đếm FLOP ở bản 1 đã chạy đúng: **0,997 GFLOP/vị trí**.


## 0. Chuẩn bị


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
%cd /content
!rm -rf chess_variant_engine
!git clone -q --depth 1 -b mcts-capacity-256 https://github.com/phuc11731510/chess_variant_engine.git
!wget -q -nc https://github.com/phuc11731510/chess_variant_engine/releases/download/v1.0.0/12bx144fx8s_12.onnx
!ls -la 12bx144fx8s_12.onnx


---
# PHẦN A — Suy luận thuần tuý  ⬅ QUAN TRỌNG NHẤT

Không MCTS, không cây. Chỉ nạp mạng và bắn tensor ngẫu nhiên vào.
Cho biết **trần tuyệt đối** của mạng này trên T4, tách khỏi mọi thứ khác.

Cell dưới sẽ gỡ `onnxruntime` cũ, cài `onnxruntime-gpu==1.20.1`, dựng
`LD_LIBRARY_PATH`, rồi chạy benchmark.

⚠ **TensorRT biên dịch kế hoạch riêng cho từng cỡ batch, 1-5 phút mỗi cỡ.**
Cell này có thể chạy 15-25 phút. Cứ để nó chạy.


In [ ]:
!bash /content/chess_variant_engine/custom_engine/scripts/bench_ort.sh \
    /content/12bx144fx8s_12.onnx --board 10


### Nếu vẫn báo `KHONG kich hoat duoc`

Chạy ô chẩn đoán dưới rồi gửi tôi output. **Đừng chạy tiếp Phần C** — số liệu
Phần A là thứ quyết định.


In [ ]:
# Chan doan: tai sao CUDA EP khong nap duoc (chay neu Phan A van ve CPU)
import glob, os, site, subprocess
print("== goi onnxruntime dang cai ==")
subprocess.run("pip list 2>/dev/null | grep -i onnxruntime", shell=True)
print("\n== thu muc thu vien nvidia tu cac goi pip cua torch ==")
n = 0
for sp in (site.getsitepackages() or []):
    for d in sorted(glob.glob(os.path.join(sp, "nvidia", "*", "lib"))):
        print("  ", d)
        n += 1
print("  tong:", n, "thu muc")
print("\n== cuDNN tim thay ==")
subprocess.run("find / -name 'libcudnn.so*' 2>/dev/null | head -5", shell=True)
print("\n== CUDA runtime ==")
subprocess.run("ls /usr/local/cuda/lib64/libcudart.so* 2>/dev/null; nvcc --version 2>/dev/null | tail -2", shell=True)


### Nếu muốn thử nhanh chỉ CUDA (bỏ TensorRT cho đỡ lâu)


In [ ]:
# Bo qua buoc cai lai (da cai o cell tren) va chi do CUDA EP
!SKIP_INSTALL=1 bash /content/chess_variant_engine/custom_engine/scripts/bench_ort.sh \
    /content/12bx144fx8s_12.onnx --board 10 --providers cuda


---
# PHẦN B — lc0 trên cùng con T4  *(tuỳ chọn)*

lc0 có **cả hai** backend: `cuda` (kernel NVIDIA viết tay) và `onnx-cuda`
(chính ORT). Tỉ số giữa chúng cho biết ORT để lại bao nhiêu hiệu năng trên bàn.

Mạng cần ~1 GFLOP để so được. Với bàn 8×8 của lc0:

```
FLOP ≈ 2304 × blocks × filters²
  12×192 → 1,02 GFLOP   ← khớp nhất với mạng 0,997 GFLOP của bạn
  15×192 → 1,27 GFLOP
  10×128 → 0,38 GFLOP   (quá nhỏ)
```


In [ ]:
import json, urllib.request
try:
    rel = json.load(urllib.request.urlopen(
        "https://api.github.com/repos/LeelaChessZero/lc0/releases/latest"))
    print("lc0 release:", rel["tag_name"])
    for a in rel["assets"]:
        print("   ", a["name"])
except Exception as e:
    print("khong truy van duoc GitHub API:", e)


In [ ]:
# Neu co asset Linux+CUDA thi sua cho khop roi bo dau # :
# !wget -q <url_asset> -O lc0.tar.gz && tar xzf lc0.tar.gz
# !wget -q <url_mang_12x192> -O lc0net.pb.gz
# import subprocess
# for be in ["cuda", "onnx-cuda", "onnx-trt", "cuda-fp16"]:
#     print("=== backend:", be, "===")
#     subprocess.run(["./lc0","benchmark","--backend="+be,
#                     "--weights=lc0net.pb.gz","--num-positions=30"])
print("lay mang o https://lczero.org/play/networks/ , chon dung blocks x filters")


---
# PHẦN C — Cắt `ev/play` trên engine

`ev/play = 1,43` nghĩa là cứ 1 playout hữu ích thì engine gửi 1,43 thế cờ lên
GPU — **30% công việc GPU không sinh ra gì**. Nguyên nhân là *collision*: khi
gom một lô, các lần đi xuống cây sau chưa biết kết quả của lần trước nên hay
rơi trúng cùng một lá chưa được đánh giá.

Cách cắt: **gom lô nhỏ hơn** → ít lần đi xuống mù hơn → ít va chạm hơn.

Trước đây giảm lô là đánh đổi đáng sợ (batch GPU nhỏ đi). Giờ ta đã **đo được
batch to không mang lại throughput**, nên giảm lô gần như miễn phí.

Cần engine đã build → ~10 phút, rồi 5 nhánh × 4 phút ≈ 25 phút.


In [ ]:
import os
if not os.path.exists("/content/chess_variant_engine/custom_engine/run.sh"):
    !bash /content/chess_variant_engine/custom_engine/scripts/colab_setup.sh > /content/build.log 2>&1
    !bash /content/chess_variant_engine/custom_engine/scripts/colab_prebuilt.sh wrap
else:
    print("engine da san sang")


In [ ]:
import subprocess, re, os

ENG  = "/content/chess_variant_engine/custom_engine/run.sh"
W    = "/content/12bx144fx8s_12.onnx"
SECS = 240

# minibatch-size=0 nghia la "dung goi y cua backend" = fixed-batch.
# Lo nho hon => it lan di xuong mu hon => it collision => ev/play thap hon.
ARMS = [
    ("C1_goc_mb16", ["--fixed-batch", "16"], []),
    ("C2_mb8",      ["--fixed-batch", "8"],  ["--search-opt", "minibatch-size=8"]),
    ("C3_mb4",      ["--fixed-batch", "4"],  ["--search-opt", "minibatch-size=4"]),
    ("C4_mb2",      ["--fixed-batch", "2"],  ["--search-opt", "minibatch-size=2"]),
    ("C5_mb8_col4", ["--fixed-batch", "8"],  ["--search-opt", "minibatch-size=8",
                                              "--search-opt", "max-collision-events=4"]),
]

rows = []
for name, fb, extra in ARMS:
    out = "/content/bench_c/" + name
    subprocess.run(["rm", "-rf", out])
    os.makedirs(out, exist_ok=True)
    cmd = ["bash", ENG, "--selfplay", "--games", "100000", "--max-seconds", str(SECS),
           "--visits", "800", "--max-moves", "400", "--temp-cutoff", "32",
           "--parallel", "4", "--provider", "cuda"] + fb + extra + \
          ["--noise-alpha", "0.15", "--show-nps", "--weights", W, "--out", out]
    print(">>> %s : %s" % (name, " ".join(fb + extra)))
    r = subprocess.run(cmd, capture_output=True, text=True)
    log = r.stdout + r.stderr
    def g(pat):
        m = re.search(pat, log)
        return m.group(1) if m else "-"
    row = (name,
           g(r"Finished (\d+)/"),
           g(r"Van/gio\s*:\s*([\d.]+)"),
           g(r"NN eval/giay\s*:\s*([\d.]+)"),
           g(r"NN eval/playout\s*:\s*([\d.]+)"),
           g(r"Batch TB moi Run\(\)\s*:\s*([\d.]+)"),
           g(r"Phi do pad\s*:\s*([\d.]+)"))
    print("    van=%s  van/gio=%s  eval/s=%s  ev/play=%s  batch=%s  pad=%s%%" % row[1:])
    rows.append(row)
    subprocess.run(["rm", "-rf", out])

print()
print("%-14s%5s%10s%10s%9s%9s%8s" % ("arm", "van", "van/gio", "eval/s", "ev/play", "batchTB", "pad%"))
print("-" * 65)
for r_ in rows:
    print("%-14s%5s%10s%10s%9s%9s%8s" % r_)


---
# Gửi lại cho tôi

1. Bảng cuối **Phần A** (CUDA EP vs TensorRT theo cỡ batch) — **ưu tiên số một**
2. Bảng cuối **Phần C** (quét ev/play)
3. Nếu Phần A vẫn lỗi: output ô chẩn đoán

Riêng bảng Phần A đã đủ để phân ba nhánh:

| Suy luận thuần | Kết luận |
|---|---|
| ≈ 2200 pos/s | T4 đúng là trần. Phần mềm hết đường |
| ≫ 2200 (2× trở lên) | Engine mất hiệu năng ở khâu điều phối CPU/GPU — sửa được, miễn phí |
| TensorRT ≫ CUDA EP | Đổi backend đáng công, giữ nguyên fp32 |
